In [ ]:
# ==========================================================
#                     IMPORTS
# ==========================================================
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
import jax.numpy as jnp
from jax import vmap
jax.config.update('jax_enable_x64', True)

print('Local devices:', jax.local_device_count(), flush=True)
print('Total devices:', jax.device_count(), flush=True)

In [ ]:
# ==========================================================
#                     NUMPYRO CONFIG
# ==========================================================
import numpyro
numpyro.enable_x64()
from numpyro.handlers import seed, trace, condition
from numpyro.infer.reparam import LocScaleReparam, TransformReparam
from numpyro.infer import HMC, HMCECS, MCMC, NUTS, SA, SVI, Trace_ELBO, init_to_value
import numpyro.distributions as dist

In [ ]:
# ==========================================================
#                     LIBRARIES
# ==========================================================
import copy
import time
import pathlib
import pickle as pk
import numpy as np
import colossus
import yaml
from deepmerge import always_merger
from astropy import constants as const
from astropy.io import fits
import matplotlib.pyplot as pl

In [ ]:
# ==========================================================
#                     GODMAX LIBRARY
# ==========================================================
import sys
curr_path        = pathlib.Path().absolute()
abs_path_src     = os.path.abspath(curr_path / '../src/')
abs_path_params  = os.path.abspath(curr_path / '../param_files/')
abs_path_data    = os.path.abspath(curr_path / '../data/')
abs_path_results = os.path.abspath(curr_path / '../results/')
sys.path.append(os.path.join(str(abs_path_src), 'arxiv'))

from godmax.base_class          import base_class
from godmax.get_radial_profiles import Profiles
from godmax.get_Pkzs            import get_Pkz
from godmax.get_Cls             import get_Cl
from godmax.get_covs            import get_cov

print('Imports done', flush=True)

In [ ]:
# ==========================================================
#   HELPER FUNCTIONS  (mirrors tSZxEuclid.godmax.sampling.model API)
# ==========================================================

def build_bin_combinations(probes, Cl_theory):
    """Return ordered list of [probe, (b1, b2)] with 1-indexed bin numbers.

    Conventions
    -----------
    yy          : (nell,)              -> [(0, 0)]                         (no bins)
    ky / gy     : (nell, nbins)        -> [(b+1, 0) for b in range(nbins)] (y has no bin)
    kk / gg     : (nell, n, n)         -> upper triangle  b2 >= b1         (symmetric)
    gk          : (nell, nl, ns)       -> all (b1+1, b2+1)                 (asymmetric)
    """
    bin_comb_all = []
    for probe in probes:
        shape = np.shape(Cl_theory[probe])
        if probe == 'yy':
            bin_comb_all.append([probe, (0, 0)])
        elif probe in ('ky', 'gy'):
            for b1 in range(shape[1]):
                bin_comb_all.append([probe, (b1 + 1, 0)])
        elif probe in ('kk', 'gg'):
            for b1 in range(shape[1]):
                for b2 in range(shape[2]):
                    if b2 >= b1:
                        bin_comb_all.append([probe, (b1 + 1, b2 + 1)])
        elif probe == 'gk':
            for b1 in range(shape[1]):
                for b2 in range(shape[2]):
                    bin_comb_all.append([probe, (b1 + 1, b2 + 1)])
    return bin_comb_all


def build_data_vector(cl_obj, bin_comb_all, nell):
    """Assemble a flat data vector from a godmax get_Cl object.

    Parameters
    ----------
    cl_obj       : get_Cl instance
    bin_comb_all : output of build_bin_combinations
    nell         : number of ell bins

    Returns
    -------
    Cl_all : jnp.ndarray, shape (len(bin_comb_all) * nell,)
    """
    Cl_all = jnp.zeros(len(bin_comb_all) * nell)
    for i, (probe, (b1, b2)) in enumerate(bin_comb_all):
        sl = slice(i * nell, (i + 1) * nell)
        if probe == 'yy':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_y_y_tot_mat)
        elif probe == 'ky':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_kappa_y_tot_mat[:, b1 - 1])
        elif probe == 'kk':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_kappa_kappa_tot_mat[:, b1 - 1, b2 - 1])
        elif probe == 'gy':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_gal_y_tot_mat[:, b1 - 1])
        elif probe == 'gg':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_gal_gal_tot_mat[:, b1 - 1, b2 - 1])
        elif probe == 'gk':
            Cl_all = Cl_all.at[sl].set(cl_obj.Cl_gal_kappa_tot_mat[:, b1 - 1, b2 - 1])
    return Cl_all


print('Helpers defined', flush=True)

In [ ]:
# =====================================================================
#                         PATHS + LOAD PARAM FILES
# =====================================================================
PARAM_DIR  = pathlib.Path(abs_path_params)
DATA_DIR   = pathlib.Path(abs_path_data)
OUTPUT_DIR = pathlib.Path(abs_path_results)

def read_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

default_data    = read_yaml(PARAM_DIR / 'params_default.yaml')
experiment_data = read_yaml(PARAM_DIR / 'DESxACT/params_v0.yaml')
merged          = always_merger.merge(default_data, experiment_data)

sim_params_dict   = merged.get('sim_params',   {})
halo_params_dict  = merged.get('halo_params',  {})
analysis_dict     = merged.get('analysis',     {})
other_params_dict = merged.get('other_params', {})

In [ ]:
# =====================================================================
#                DEFINE COSMOLOGY
# =====================================================================
from jax_cosmo import Cosmology

cosmo_params = sim_params_dict['cosmo']
cosmo_jax = Cosmology(
    Omega_c = cosmo_params['Om0'] - cosmo_params['Ob0'],
    Omega_b = cosmo_params['Ob0'],
    h       = cosmo_params['H0'] / 100.0,
    sigma8  = cosmo_params['sigma8'],
    n_s     = cosmo_params['ns'],
    Omega_k = 0.0,
    w0      = cosmo_params['w0'],
    wa      = 0.0,
)
print('Cosmology:', cosmo_params)

In [ ]:
# =====================================================================
#                DEFINE ELL ARRAYS
# =====================================================================
lmin, lmax, dl_log = 10.0, 10000.0, 0.23025851

l_edges  = np.exp(np.arange(np.log(lmin), np.log(lmax), dl_log))
dl_array = jnp.array(l_edges[1:] - l_edges[:-1])
l_eff    = jnp.array((l_edges[1:] + l_edges[:-1]) / 2.0)

halo_params_dict['ell_array']    = l_eff
analysis_dict['l_array_survey']  = l_eff
analysis_dict['dl_array_survey'] = dl_array

nell = len(l_eff)
print(f'ell range: [{float(l_eff[0]):.1f}, {float(l_eff[-1]):.1f}],  nell={nell}')

In [ ]:
# ==========================================================
#                SOURCE N(Z) BINS
# ==========================================================
fits_file    = DATA_DIR / 'DESxACT/2pt_NG_final_2ptunblind_02_26_21_wnz_maglim_covupdate.fits'
df           = fits.open(fits_file)
nbins_source = 4

z_src  = df['nz_source'].data['Z_MID']
nz_src = {ji: np.maximum(df['nz_source'].data[f'BIN{ji+1}'], 1e-4)
          for ji in range(nbins_source)}

nz_source_info = {'nbins': nbins_source, 'z_array_source': z_src}
for ji in range(nbins_source):
    nz_source_info[f'nz{ji}'] = nz_src[ji]
analysis_dict['nz_source_info_dict'] = nz_source_info

other_params_dict['Delta_z_bias_array']    = np.zeros(nbins_source)
other_params_dict['mult_shear_bias_array'] = np.zeros(nbins_source)
print(f'Source bins: {nbins_source}  z in [{z_src[0]:.2f}, {z_src[-1]:.2f}]')

In [ ]:
# ==========================================================
#                LENS N(Z) BINS
# ==========================================================
# Using the same DES source bins as lens here.
# Replace with your own lens catalog as needed.
nbins_lens = 4

nz_lens_info = {'nbins_lens': nbins_lens, 'z_array_lens': z_src.copy()}
for ji in range(nbins_lens):
    nz_lens_info[f'nz{ji}'] = nz_src[ji]
analysis_dict['nz_lens_info_dict'] = nz_lens_info

print(f'Lens bins:   {nbins_lens}')
print('N(z) setup done', flush=True)

In [ ]:
# ==========================================================
#      SYMBOLIC APPROXIMATIONS (optional — faster / differentiable)
# ==========================================================
# Uncomment to enable:
# analysis_dict['symbolic_pk']  = True
# analysis_dict['symbolic_hmf'] = True

In [ ]:
# ==========================================================
#                     PROBES
# ==========================================================
probes_analysis = ['yy', 'ky', 'kk', 'gy', 'gg', 'gk']
labels = [
    r'$\langle yy \rangle$',
    r'$\langle \kappa y \rangle$',
    r'$\langle \kappa\kappa \rangle$',
    r'$\langle gy \rangle$',
    r'$\langle gg \rangle$',
    r'$\langle g\kappa \rangle$',
]

In [ ]:
# =====================================================================
#                COMPUTE THEORY = DATA VEC
# =====================================================================
t0 = time.time()
get_cl = get_Cl(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict)
print(f'Theory computed in {time.time()-t0:.1f}s', flush=True)

In [ ]:
Cl_theory = {
    'yy': get_cl.Cl_y_y_tot_mat,
    'ky': get_cl.Cl_kappa_y_tot_mat,
    'kk': get_cl.Cl_kappa_kappa_tot_mat,
    'gy': get_cl.Cl_gal_y_tot_mat,
    'gg': get_cl.Cl_gal_gal_tot_mat,
    'gk': get_cl.Cl_gal_kappa_tot_mat,
}

bin_comb_all = build_bin_combinations(probes_analysis, Cl_theory)
data_vec     = build_data_vector(get_cl, bin_comb_all, nell)

os.makedirs(OUTPUT_DIR / 'tests/mock-data', exist_ok=True)
np.savetxt(OUTPUT_DIR / 'tests/mock-data/mock-Cls.txt', data_vec)
np.savetxt(OUTPUT_DIR / 'tests/mock-data/mock-ell.txt', analysis_dict['l_array_survey'])
print('Mock data computed', flush=True)

In [ ]:
pl.figure(figsize=(10, 6))
pl.semilogy(np.abs(data_vec))
# Mark probe boundaries
seen, cursor = {}, 0
for probe, _ in bin_comb_all:
    if probe not in seen:
        pl.axvline(cursor, color='grey', lw=0.8, ls='--', alpha=0.7)
        pl.text(cursor + nell * 0.4, np.abs(data_vec).max() * 0.3,
                probe, fontsize=10, color='grey')
        seen[probe] = True
    cursor += nell
pl.xlabel('Data vector index', fontsize=12)
pl.ylabel(r'$|C_\ell|$', fontsize=12)
pl.title('Flat data vector — all probes concatenated', fontsize=12)
pl.grid(True, alpha=0.3)
pl.tight_layout()
pl.show()

In [ ]:
# =====================================================================
#             tSZ NOISE POWER SPECTRUM (null here — needed for get_cov)
# =====================================================================
yy_noise_file = OUTPUT_DIR / 'tests/mock-data/yy-false-Cl.txt'
np.savetxt(
    yy_noise_file,
    np.column_stack([
        np.array(analysis_dict['l_array_survey']),
        1e-25 * np.ones(nell),
    ]),
)
analysis_dict['yy_noise_ell_fname'] = str(yy_noise_file)
print(f'yy noise written to {yy_noise_file}')